In [1]:
!git clone https://github.com/pablovin/ChefsHatGYM.git
%cd ChefsHatGYM
!pip install -e .

Cloning into 'ChefsHatGYM'...
remote: Enumerating objects: 2545, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 2545 (delta 155), reused 134 (delta 117), pack-reused 2335 (from 3)
Receiving objects: 100% (2545/2545), 141.01 MiB | 16.12 MiB/s, done.
Resolving deltas: 100% (1420/1420), done.
Updating files: 100% (144/144), done.
/content/ChefsHatGYM
Obtaining file:///content/ChefsHatGYM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build editable did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build editable ... error
error: subprocess-exited-with-error

× Getting requirements to build editable did not run successfully.
│ exit code: 1
╰─> See above for output.

In [2]:
%cd src

/content/ChefsHatGYM/src


In [3]:
!pip install nest_asyncio
import nest_asyncio
nest_asyncio.apply()

In [4]:
import os
import asyncio
import ast
import pandas as pd

from agents.random_agent import RandomAgent
from agents.agent_dqn import DQNAgent
from rooms.room import Room

/content/ChefsHatGYM/src/core/logging/room_logger.py:57: SyntaxWarning: invalid escape sequence '\|'
  | __|_ _|_ _| \| | /_\


In [5]:
async def run_experiment(training, model_path, matches, output_folder, reward_mode="original"):

    room = Room(
        run_remote_room=False,
        room_name="SparseRewardRoom",
        max_matches=matches,
        output_folder=output_folder,
        save_game_dataset=True,
        save_logs_game=False,
        save_logs_room=False,
    )

    # 3 Random agents
    for i in range(3):
        room.connect_player(
            RandomAgent(
                name=f"Random{i}",
                log_directory=room.room_dir,
                verbose_log=False
            )
        )

    # DQN Agent
    agent = DQNAgent(
        name="DQN",
        train=training,
        log_directory=room.room_dir,
        model_path=model_path,
        load_model=not training,
    )

    # -------- Reward Shaping Logic --------
    if reward_mode == "shaped":

        original_remember = agent.remember

        def shaped_remember(state, possible_actions, action, reward,
                            next_state, next_possible_actions, done):

            if done:
                reward = reward * 2.0
            else:
                reward = reward * 0.2

            original_remember(state, possible_actions, action,
                              reward, next_state,
                              next_possible_actions, done)

        agent.remember = shaped_remember

    room.connect_player(agent)

    await room.run()

    return room

In [7]:
print("Training Original Reward...")
await run_experiment(True, "outputs_original/model_original.h5", 100, "outputs_original", "original")

print("Testing Original Reward...")
await run_experiment(False, "outputs_original/model_original.h5", 100, "outputs_original_test", "original")

Training Original Reward...
Log_directory: outputs_original/SparseRewardRoom_20260303_125840_314408
Log_directory: outputs_original/SparseRewardRoom_20260303_125840_314408
Log_directory: outputs_original/SparseRewardRoom_20260303_125840_314408
Log_directory: outputs_original/SparseRewardRoom_20260303_125840_314408


Testing Original Reward...
Log_directory: outputs_original_test/SparseRewardRoom_20260303_130020_794811
Log_directory: outputs_original_test/SparseRewardRoom_20260303_130020_794811
Log_directory: outputs_original_test/SparseRewardRoom_20260303_130020_794811
Log_directory: outputs_original_test/SparseRewardRoom_20260303_130020_794811


In [8]:
print("Training Shaped Reward...")
await run_experiment(True, "outputs_shaped/model_shaped.h5", 100, "outputs_shaped", "shaped")

print("Testing Shaped Reward...")
await run_experiment(False, "outputs_shaped/model_shaped.h5", 100, "outputs_shaped_test", "shaped")

Training Shaped Reward...
Log_directory: outputs_shaped/SparseRewardRoom_20260303_130448_201543
Log_directory: outputs_shaped/SparseRewardRoom_20260303_130448_201543
Log_directory: outputs_shaped/SparseRewardRoom_20260303_130448_201543
Log_directory: outputs_shaped/SparseRewardRoom_20260303_130448_201543


Testing Shaped Reward...
Log_directory: outputs_shaped_test/SparseRewardRoom_20260303_130631_430542
Log_directory: outputs_shaped_test/SparseRewardRoom_20260303_130631_430542
Log_directory: outputs_shaped_test/SparseRewardRoom_20260303_130631_430542
Log_directory: outputs_shaped_test/SparseRewardRoom_20260303_130631_430542


In [9]:
def calculate_winrate(folder):

    import glob
    import pandas as pd
    import ast

    path = glob.glob(folder + "/*/dataset/game_dataset.pkl.csv")[0]

    df = pd.read_csv(path, index_col=0)
    end_matches = df[df["Action_Type"] == "END_MATCH"]

    wins = 0
    total = len(end_matches)

    for s in end_matches["Match_Score"]:
        score_dict = ast.literal_eval(s)

        # In Chef's Hat lower score = better position
        if score_dict["DQN"] == min(score_dict.values()):
            wins += 1

    print("Total Matches:", total)
    print("Wins (1st place):", wins)
    print("Win Rate:", wins / total)


print("Original Reward Results:")
calculate_winrate("outputs_original_test")

print("\nShaped Reward Results:")
calculate_winrate("outputs_shaped_test")

Original Reward Results:
Total Matches: 100
Wins (1st place): 22
Win Rate: 0.22

Shaped Reward Results:
Total Matches: 100
Wins (1st place): 14
Win Rate: 0.14


In [10]:
from google.colab import files
import glob

# Download original test dataset
orig_path = glob.glob("outputs_original_test/*/dataset/game_dataset.pkl.csv")[0]
files.download(orig_path)

# Download shaped test dataset
shaped_path = glob.glob("outputs_shaped_test/*/dataset/game_dataset.pkl.csv")[0]
files.download(shaped_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
files.download("outputs_original/model_original.h5")
files.download("outputs_shaped/model_shaped.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>